# Normalización Robusta de Direcciones

Limpia y normaliza direcciones de un CSV robusto:
1. Elimina especificaciones de departamentos, torres y números de oficina
2. Valida formato usando regex complejos
3. Extrae calles y números de forma inteligente
4. Formatea direcciones con mayúsculas y correcciones de acentos
5. Prepara direcciones para geocodificación agregando ", Santiago, Chile"
6. Genera un CSV normalizado con validaciones de calidad

In [ ]:
import re
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
INPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\ventas_ficticias_santiago_202612.csv"
OUTPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_normalizadas.csv"

# =========================================================
# UTILIDADES
# =========================================================
def normalizar_espacios(texto):
    return re.sub(r"\s+", " ", str(texto)).strip()

def limpiar_comillas(texto):
    texto = str(texto)
    texto = texto.replace('""', '"')
    texto = texto.strip().strip('"').strip("'")
    texto = texto.replace('"', ' ')
    texto = texto.replace("'", " ")
    return normalizar_espacios(texto)

def parsear_linea(linea):
    """
    Reconstruye la fila aunque la dirección tenga comas.
    Espera:
    RUT,Nombre cliente,Dirección cliente,Fecha de Pedido,Número de Orden,Monto Pedido,Fecha de despacho Solicitada
    """
    s = linea.strip()

    if s.startswith('"') and s.endswith('"'):
        s = s[1:-1]
        s = s.replace('""', '"')

    patron = re.compile(
        r'^(.*?),(.*?),(.*?),(20\d{2}-\d{2}-\d{2}),(ORD-[^,]+),([^,]+),(20\d{2}-\d{2}-\d{2})$'
    )

    m = patron.match(s)
    if not m:
        return None

    rut, nombre, direccion, fecha_pedido, numero_orden, monto, fecha_despacho = m.groups()

    direccion = limpiar_comillas(direccion)

    return {
        "RUT": normalizar_espacios(rut),
        "Nombre cliente": normalizar_espacios(nombre),
        "Dirección cliente": direccion,
        "Fecha de Pedido": normalizar_espacios(fecha_pedido),
        "Número de Orden": normalizar_espacios(numero_orden),
        "Monto Pedido": normalizar_espacios(monto),
        "Fecha de despacho Solicitada": normalizar_espacios(fecha_despacho),
    }

def cargar_csv_robusto(path_csv):
    filas = []
    filas_malas = []

    with open(path_csv, "r", encoding="utf-8-sig", errors="replace") as f:
        next(f)  # saltar encabezado
        for i, linea in enumerate(f, start=2):
            fila = parsear_linea(linea)
            if fila is None:
                filas_malas.append({"linea": i, "contenido": linea.strip()})
            else:
                filas.append(fila)

    df = pd.DataFrame(filas)
    df_bad = pd.DataFrame(filas_malas)
    return df, df_bad

# =========================================================
# LIMPIEZA DE DIRECCIONES
# =========================================================
PATRON_DEPTO = re.compile(
    r'\b(DEPARTAMENTO|DEPTO|DPTO|DTO)\b\.?\s*[A-Z0-9\-]+',
    flags=re.IGNORECASE
)

COMUNAS = [
    "CERRILLOS", "CERRO NAVIA", "CONCHALI", "CONCHALÍ", "EL BOSQUE",
    "ESTACION CENTRAL", "ESTACIÓN CENTRAL", "HUECHURABA", "INDEPENDENCIA",
    "LA CISTERNA", "LA FLORIDA", "LA GRANJA", "LA PINTANA", "LA REINA",
    "LAS CONDES", "LO BARNECHEA", "LO ESPEJO", "LO PRADO", "MACUL",
    "MAIPU", "MAIPÚ", "NUNOA", "NUÑOA", "ÑUÑOA", "PEDRO AGUIRRE CERDA",
    "PENALOLEN", "PEÑALOLÉN", "PEÑALOLEN", "PROVIDENCIA", "PUDAHUEL",
    "PUENTE ALTO", "QUILICURA", "QUINTA NORMAL", "RECOLETA", "RENCA",
    "SAN BERNARDO", "SAN JOAQUIN", "SAN JOAQUÍN", "SAN MIGUEL",
    "SAN RAMON", "SAN RAMÓN", "SANTIAGO", "VITACURA"
]

def quitar_ruido_final(s):
    s = normalizar_espacios(s)

    cambios = True
    while cambios:
        cambios = False

        s_nuevo = re.sub(r'\b(CHILE|STGO|SANTIAGO)\b\s*$', '', s, flags=re.IGNORECASE).strip()
        if s_nuevo != s:
            s = normalizar_espacios(s_nuevo)
            cambios = True

        for comuna in COMUNAS:
            patron = rf'\b{re.escape(comuna)}\b\s*$'
            s_nuevo = re.sub(patron, '', s, flags=re.IGNORECASE).strip()
            if s_nuevo != s:
                s = normalizar_espacios(s_nuevo)
                cambios = True

    return s

def normalizar_abreviaturas_calle(s):
    s = normalizar_espacios(s)
    s = re.sub(r'(?i)\bav\s*\.\s*', 'Av. ', s)
    s = re.sub(r'(?i)\bav\s+', 'Av. ', s)
    s = re.sub(r'\.\.+', '.', s)
    s = s.replace("Av..", "Av.")
    s = s.replace("Av. .", "Av. ")
    return normalizar_espacios(s)

def limpiar_direccion_base(direccion):
    s = limpiar_comillas(direccion)
    s = normalizar_espacios(s)

    s = s.replace(",", " ")
    s = PATRON_DEPTO.sub("", s)
    s = s.replace('"', ' ')
    s = s.replace("'", " ")
    s = normalizar_espacios(s)

    s = quitar_ruido_final(s)
    s = normalizar_abreviaturas_calle(s)

    return s

def formatear_calle(calle):
    calle = normalizar_abreviaturas_calle(calle)
    calle = calle.title()

    reemplazos = {
        " De ": " de ",
        " Del ": " del ",
        " Y ": " y ",
    }
    for a, b in reemplazos.items():
        calle = calle.replace(a, b)

    correcciones = {
        "Nunoa": "Ñuñoa",
        "Penalolen": "Peñalolén",
        "Jose": "José",
        "Vicuna": "Vicuña",
        "Compania": "Compañía",
        "Jesus": "Jesús",
        "Joaquin": "Joaquín",
        "Ramon": "Ramón",
        "Irarrazaval": "Irarrázaval",
        "Americo": "Américo",
        "Morande": "Morandé",
    }
    for a, b in correcciones.items():
        calle = calle.replace(a, b)

    return normalizar_espacios(calle)

def extraer_calle_numero(direccion):
    """
    Busca la primera estructura tipo:
    texto + numero
    """
    s = limpiar_direccion_base(direccion)

    m = re.search(r'(.+?)\s+(\d{1,6})\b', s)
    if not m:
        return None, None

    calle = formatear_calle(m.group(1))
    numero = m.group(2)

    return calle, numero

def construir_direccion_normalizada(calle, numero):
    if calle is None or numero is None or pd.isna(calle) or pd.isna(numero):
        return None
    return f"{calle} {numero}"

# =========================================================
# COMPROBACIONES
# =========================================================
def tiene_comillas(texto):
    if pd.isna(texto):
        return False
    t = str(texto)
    return ('"' in t) or ("'" in t)

def contiene_depto(texto):
    if pd.isna(texto):
        return False
    return bool(re.search(r'\b(DEPARTAMENTO|DEPTO|DPTO|DTO)\b', str(texto), flags=re.IGNORECASE))

def parece_direccion_valida(texto):
    """
    Valida algo tipo 'Texto 1234'
    """
    if pd.isna(texto):
        return False
    return bool(re.match(r'^.+\s+\d{1,6}$', str(texto).strip()))

def resumen_comprobacion(df):
    print("\n" + "=" * 60)
    print("COMPROBACIÓN DE CALIDAD")
    print("=" * 60)

    total = len(df)
    sin_calle = df["calle"].isna().sum()
    sin_numero = df["numero"].isna().sum()
    sin_normalizada = df["direccion_normalizada"].isna().sum()

    con_comillas = df["direccion_normalizada"].apply(tiene_comillas).sum()
    con_depto = df["direccion_limpia_base"].apply(contiene_depto).sum()
    formato_invalido = (~df["direccion_normalizada"].apply(parece_direccion_valida) & df["direccion_normalizada"].notna()).sum()

    print(f"Total filas: {total}")
    print(f"Sin calle: {sin_calle}")
    print(f"Sin número: {sin_numero}")
    print(f"Sin dirección normalizada: {sin_normalizada}")
    print(f"Direcciones normalizadas con comillas: {con_comillas}")
    print(f"Direcciones limpias que aún contienen 'depto': {con_depto}")
    print(f"Direcciones normalizadas con formato inesperado: {formato_invalido}")

    print("\nMuestra de direcciones normalizadas:")
    print(df[["Dirección cliente", "direccion_normalizada"]].head(15).to_string(index=False))

    print("\nFilas problemáticas:")
    problemas = df[
        df["direccion_normalizada"].isna()
        | df["direccion_normalizada"].apply(tiene_comillas)
        | (~df["direccion_normalizada"].apply(parece_direccion_valida) & df["direccion_normalizada"].notna())
    ][["Dirección cliente", "direccion_limpia_base", "calle", "numero", "direccion_normalizada"]]

    if len(problemas) == 0:
        print("No se detectaron filas problemáticas en la comprobación básica.")
    else:
        print(problemas.head(20).to_string(index=False))

# =========================================================
# PROCESO
# =========================================================
df, df_bad = cargar_csv_robusto(INPUT_CSV)

print("Filas cargadas:", len(df))
print("Filas no parseadas:", len(df_bad))

if len(df_bad) > 0:
    print("\nPrimeras filas no parseadas:")
    print(df_bad.head(10).to_string(index=False))

df["direccion_limpia_base"] = df["Dirección cliente"].apply(limpiar_direccion_base)

df[["calle", "numero"]] = df["Dirección cliente"].apply(
    lambda x: pd.Series(extraer_calle_numero(x))
)

df["direccion_normalizada"] = df.apply(
    lambda row: construir_direccion_normalizada(row["calle"], row["numero"]),
    axis=1
)

df["query_geocoding"] = df["direccion_normalizada"].apply(
    lambda x: f"{x}, Santiago, Chile" if pd.notna(x) else None
)

# =========================================================
# COMPROBACIÓN FINAL
# =========================================================
resumen_comprobacion(df)

# =========================================================
# GUARDAR
# =========================================================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nArchivo generado:")
print(OUTPUT_CSV)

print("\nColumnas principales:")
print(
    df[
        [
            "Dirección cliente",
            "direccion_limpia_base",
            "calle",
            "numero",
            "direccion_normalizada",
            "query_geocoding"
        ]
    ].head(20).to_string(index=False)
)

# Geocodificación de Direcciones 1 al 2500 con Geoapify

Geocodifica direcciones del rango 1 al 2500 usando la API de Geoapify. Realiza consultas a la API con pausa de 0.25 segundos entre cada una para evitar saturación. 

In [ ]:
import time
import re
import requests
import pandas as pd

INPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_normalizadas.csv"
OUTPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_1_2500_geoapify.csv"
API_KEY = "fb9ac209cf6648b3b8509e676abee204"

SLEEP_SECONDS = 0.25

def geocode_geoapify(query, api_key, session):
    url = "https://api.geoapify.com/v1/geocode/search"
    params = {
        "text": query,
        "apiKey": api_key,
        "lang": "es",
        "limit": 1,
        "filter": "countrycode:cl",
        "format": "json"
    }

    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    if not results:
        return {
            "encontrada": False,
            "formatted": None,
            "lat": None,
            "lon": None,
            "estado": "no_encontrado"
        }

    res = results[0]
    return {
        "encontrada": True,
        "formatted": res.get("formatted"),
        "lat": res.get("lat"),
        "lon": res.get("lon"),
        "estado": "ok"
    }

def clasificar_match(direccion, formatted, estado, encontrada):
    if not encontrada or str(estado).lower() != "ok":
        return "no_encontrada"

    direccion = str(direccion).strip().lower()
    formatted = str(formatted).strip().lower()

    numero_match = re.search(r'(\d{1,6})', direccion)
    numero = numero_match.group(1) if numero_match else None

    palabras = [p for p in re.findall(r"[a-záéíóúñ0-9]+", direccion) if len(p) > 2]
    coincidencias = sum(1 for p in palabras if p in formatted)

    tiene_numero = numero is not None and numero in formatted

    if tiene_numero and coincidencias >= 2:
        return "buena"

    if tiene_numero or coincidencias >= 2:
        return "aceptable"

    return "dudosa"

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

if "query_geocoding" not in df.columns:
    raise ValueError("No existe la columna 'query_geocoding' en el CSV.")

# Solo filas con query válida
df = df[df["query_geocoding"].notna()].copy()

# Primeras 2500
df = df.iloc[:2500].copy()

print("Filas a procesar:", len(df))

cache = {}
queries_unicas = df["query_geocoding"].dropna().unique().tolist()

print("Queries únicas a procesar:", len(queries_unicas))

with requests.Session() as session:
    for i, q in enumerate(queries_unicas, start=1):
        print(f"[{i}/{len(queries_unicas)}] {q}")
        try:
            cache[q] = geocode_geoapify(q, API_KEY, session)
        except Exception as e:
            cache[q] = {
                "encontrada": False,
                "formatted": None,
                "lat": None,
                "lon": None,
                "estado": f"error: {e}"
            }
        time.sleep(SLEEP_SECONDS)

df["encontrada"] = df["query_geocoding"].map(lambda q: cache[q]["encontrada"])
df["formatted"] = df["query_geocoding"].map(lambda q: cache[q]["formatted"])
df["lat"] = df["query_geocoding"].map(lambda q: cache[q]["lat"])
df["lon"] = df["query_geocoding"].map(lambda q: cache[q]["lon"])
df["estado"] = df["query_geocoding"].map(lambda q: cache[q]["estado"])

df["calidad_match"] = df.apply(
    lambda row: clasificar_match(
        row["direccion_normalizada"],
        row["formatted"],
        row["estado"],
        row["encontrada"]
    ),
    axis=1
)

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nArchivo generado:")
print(OUTPUT_CSV)

print("\nResumen estado:")
print(df["estado"].value_counts(dropna=False))

print("\nResumen calidad_match:")
print(df["calidad_match"].value_counts(dropna=False))

# Geocodificación de Direcciones 2501 Hacia el Final con Geoapify

Geocodifica el resto de direcciones desde 2501 hasta el final del dataset usando la API de Geoapify. Similar al procesamiento anterior pero para el rango complementario. Asegura que todas las direcciones tengan coordenadas válidas antes de procesarlas.

In [ ]:
import time
import re
import requests
import pandas as pd

INPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_normalizadas.csv"
OUTPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_2501_fin_geoapify.csv"
API_KEY = "fb9ac209cf6648b3b8509e676abee204"

SLEEP_SECONDS = 0.25

def geocode_geoapify(query, api_key, session):
    url = "https://api.geoapify.com/v1/geocode/search"
    params = {
        "text": query,
        "apiKey": api_key,
        "lang": "es",
        "limit": 1,
        "filter": "countrycode:cl",
        "format": "json"
    }

    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    if not results:
        return {
            "encontrada": False,
            "formatted": None,
            "lat": None,
            "lon": None,
            "estado": "no_encontrado"
        }

    res = results[0]
    return {
        "encontrada": True,
        "formatted": res.get("formatted"),
        "lat": res.get("lat"),
        "lon": res.get("lon"),
        "estado": "ok"
    }

def clasificar_match(direccion, formatted, estado, encontrada):
    if not encontrada or str(estado).lower() != "ok":
        return "no_encontrada"

    direccion = str(direccion).strip().lower()
    formatted = str(formatted).strip().lower()

    numero_match = re.search(r'(\d{1,6})', direccion)
    numero = numero_match.group(1) if numero_match else None

    palabras = [p for p in re.findall(r"[a-záéíóúñ0-9]+", direccion) if len(p) > 2]
    coincidencias = sum(1 for p in palabras if p in formatted)

    tiene_numero = numero is not None and numero in formatted

    if tiene_numero and coincidencias >= 2:
        return "buena"

    if tiene_numero or coincidencias >= 2:
        return "aceptable"

    return "dudosa"

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

if "query_geocoding" not in df.columns:
    raise ValueError("No existe la columna 'query_geocoding' en el CSV.")

# Solo filas con query válida
df = df[df["query_geocoding"].notna()].copy()

# Desde la fila 2501 hasta el final
df = df.iloc[2500:].copy()

print("Filas a procesar:", len(df))

cache = {}
queries_unicas = df["query_geocoding"].dropna().unique().tolist()

print("Queries únicas a procesar:", len(queries_unicas))

with requests.Session() as session:
    for i, q in enumerate(queries_unicas, start=1):
        print(f"[{i}/{len(queries_unicas)}] {q}")
        try:
            cache[q] = geocode_geoapify(q, API_KEY, session)
        except Exception as e:
            cache[q] = {
                "encontrada": False,
                "formatted": None,
                "lat": None,
                "lon": None,
                "estado": f"error: {e}"
            }
        time.sleep(SLEEP_SECONDS)

df["encontrada"] = df["query_geocoding"].map(lambda q: cache[q]["encontrada"])
df["formatted"] = df["query_geocoding"].map(lambda q: cache[q]["formatted"])
df["lat"] = df["query_geocoding"].map(lambda q: cache[q]["lat"])
df["lon"] = df["query_geocoding"].map(lambda q: cache[q]["lon"])
df["estado"] = df["query_geocoding"].map(lambda q: cache[q]["estado"])

df["calidad_match"] = df.apply(
    lambda row: clasificar_match(
        row["direccion_normalizada"],
        row["formatted"],
        row["estado"],
        row["encontrada"]
    ),
    axis=1
)

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\nArchivo generado:")
print(OUTPUT_CSV)

print("\nResumen estado:")
print(df["estado"].value_counts(dropna=False))

print("\nResumen calidad_match:")
print(df["calidad_match"].value_counts(dropna=False))

# Unión de Archivos Geocodificados

Combina los dos archivos geocodificados (direcciones 1-2500 y 2501-fin) en un único CSV unificado que contiene todas las direcciones con sus coordenadas y estados de geocodificación.

In [ ]:
import pandas as pd

CSV_1 = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_1_2500_geoapify.csv"
CSV_2 = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_2501_fin_geoapify.csv"
OUTPUT_CSV = r"C:\Users\thoma\OneDrive\Desktop\UDD\Capstone analytics\Datos\direcciones_geoapify_unido.csv"

df1 = pd.read_csv(CSV_1, encoding="utf-8-sig")
df2 = pd.read_csv(CSV_2, encoding="utf-8-sig")

df_final = pd.concat([df1, df2], ignore_index=True)

df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Archivo unido generado en:")
print(OUTPUT_CSV)
print("\nTotal filas:", len(df_final))

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

# =========================================================
# CONFIG
# =========================================================
ruta_csv = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\nodos_master.csv"
ruta_excel_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\matriz_distancias_nodos_master.xlsx"

# =========================================================
# FUNCION HAVERSINE
# =========================================================
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calcula distancia en kilómetros entre dos puntos geográficos.
    """
    R = 6371.0  # radio de la Tierra en km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * asin(sqrt(a))

    return R * c

# =========================================================
# 1) LEER CSV
# =========================================================
df = pd.read_csv(ruta_csv)

# Asegurar tipos
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Filtrar nodos con coordenadas válidas
df_validos = df[
    df["lat"].notna() &
    df["lon"].notna()
].copy()

# Si existe la columna tiene_coordenadas, la usamos también
if "tiene_coordenadas" in df_validos.columns:
    df_validos["tiene_coordenadas"] = df_validos["tiene_coordenadas"].astype(str).str.upper()
    df_validos = df_validos[df_validos["tiene_coordenadas"] == "TRUE"].copy()

# =========================================================
# 2) USAR NODE_ID COMO ETIQUETA
# =========================================================
df_validos["etiqueta_direccion"] = df_validos["node_id"].astype(int).astype(str)

# Evitar duplicados en etiquetas por si dos filas terminan con el mismo nombre
# Si hay duplicados, se agrega node_id al final
duplicadas = df_validos["etiqueta_direccion"].duplicated(keep=False)
df_validos.loc[duplicadas, "etiqueta_direccion"] = (
    df_validos.loc[duplicadas, "etiqueta_direccion"]
    + " | node_id="
    + df_validos.loc[duplicadas, "node_id"].astype(str)
)

# =========================================================
# 3) IDENTIFICAR DEPOSITO
# =========================================================
depositos = df_validos[df_validos["tipo"].astype(str).str.lower() == "deposito"].copy()

if len(depositos) == 0:
    raise ValueError("No se encontró ninguna fila con tipo='deposito'.")

if len(depositos) > 1:
    print("Advertencia: se encontró más de un depósito. Se usará el primero.")

etiqueta_deposito = depositos.iloc[0]["etiqueta_direccion"]

# =========================================================
# 4) ORDENAR PARA DEJAR EL DEPOSITO PRIMERO
# =========================================================
df_validos["es_deposito"] = df_validos["tipo"].astype(str).str.lower() == "deposito"
df_validos = df_validos.sort_values(by=["es_deposito", "node_id"], ascending=[False, True]).reset_index(drop=True)

# =========================================================
# 5) CONSTRUIR MATRIZ DE DISTANCIAS
# =========================================================
n = len(df_validos)
coords = df_validos[["lat", "lon"]].to_numpy()
labels = df_validos["etiqueta_direccion"].tolist()

matriz = np.zeros((n, n), dtype=float)

for i in range(n):
    lat1, lon1 = coords[i]
    for j in range(i + 1, n):
        lat2, lon2 = coords[j]
        d = haversine_km(lat1, lon1, lat2, lon2)
        matriz[i, j] = d
        matriz[j, i] = d

df_matriz = pd.DataFrame(matriz, index=labels, columns=labels)

# Opcional: redondear
df_matriz = df_matriz.round(3)

# =========================================================
# 6) EXPORTAR A EXCEL
# =========================================================
with pd.ExcelWriter(ruta_excel_salida, engine="openpyxl") as writer:
    df_matriz.to_excel(writer, sheet_name="matriz_distancias")
    
    # Hoja adicional con metadata útil
    df_meta = df_validos[[
        "node_id", "direccion_original", "direccion_normalizada",
        "lat", "lon", "estado", "calidad_match", "tipo", "etiqueta_direccion"
    ]].copy()
    df_meta.to_excel(writer, sheet_name="nodos_usados", index=False)

# =========================================================
# 7) FORMATO EXCEL: RESALTAR DEPOSITO
# =========================================================
wb = load_workbook(ruta_excel_salida)
ws = wb["matriz_distancias"]

fill_deposito = PatternFill(fill_type="solid", fgColor="FFF2CC")  # amarillo suave
font_negrita = Font(bold=True)
align_center = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Encabezados de columnas están en fila 1 desde columna 2
# Encabezados de filas están en columna 1 desde fila 2

# Ajustar ancho de columnas
for col in range(1, ws.max_column + 1):
    col_letter = get_column_letter(col)
    if col == 1:
        ws.column_dimensions[col_letter].width = 45
    else:
        ws.column_dimensions[col_letter].width = 18

# Ajustar alto de la fila de encabezado
ws.row_dimensions[1].height = 30

# Formato general encabezados
for cell in ws[1]:
    cell.font = font_negrita
    cell.alignment = align_center

for row in range(2, ws.max_row + 1):
    ws.cell(row=row, column=1).font = font_negrita

# Buscar columna del deposito
col_deposito = None
for col in range(2, ws.max_column + 1):
    if ws.cell(row=1, column=col).value == etiqueta_deposito:
        col_deposito = col
        break

# Buscar fila del deposito
fila_deposito = None
for row in range(2, ws.max_row + 1):
    if ws.cell(row=row, column=1).value == etiqueta_deposito:
        fila_deposito = row
        break

# Resaltar fila y columna del deposito
if col_deposito is not None:
    for row in range(1, ws.max_row + 1):
        ws.cell(row=row, column=col_deposito).fill = fill_deposito

if fila_deposito is not None:
    for col in range(1, ws.max_column + 1):
        ws.cell(row=fila_deposito, column=col).fill = fill_deposito

# También resaltar la celda esquina superior izquierda
ws["A1"] = "node_id"
ws["A1"].font = font_negrita
ws["A1"].fill = fill_deposito
ws["A1"].alignment = align_center

wb.save(ruta_excel_salida)

print("Matriz de distancias generada correctamente en:")
print(ruta_excel_salida)
print(f"Depósito destacado: {etiqueta_deposito}")

Matriz de distancias generada correctamente en:
C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\matriz_distancias_nodos_master.xlsx
Depósito destacado: 0


# Cruze info

In [3]:
import pandas as pd
import numpy as np
import os
import unicodedata

# =========================================================
# RUTAS
# =========================================================
ruta_direcciones = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\raw\direcciones_geoapify_unido.csv"
ruta_nodos = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\nodos_master.csv"
ruta_detalle = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\raw\detalle_pedidos_santiago_202612.xlsx"

carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed"
os.makedirs(carpeta_salida, exist_ok=True)

ruta_salida_maestra = os.path.join(carpeta_salida, "pedidos_maestro_clusterizacion.csv")
ruta_salida_sin_nodo = os.path.join(carpeta_salida, "pedidos_sin_node_id.csv")
ruta_salida_sin_detalle = os.path.join(carpeta_salida, "pedidos_sin_detalle.csv")
ruta_salida_resumen = os.path.join(carpeta_salida, "resumen_construccion_maestra.xlsx")

# =========================================================
# FUNCIONES AUXILIARES
# =========================================================
def normalizar_texto(x):
    """
    Normaliza texto para merges más robustos:
    - pasa a string
    - trim
    - mayúsculas
    - quita tildes
    - compacta espacios
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = " ".join(x.split())
    return x


def convertir_fecha(s):
    return pd.to_datetime(s, errors="coerce")


# =========================================================
# 1) LEER ARCHIVOS
# =========================================================
print("Leyendo archivos...")

df_dir = pd.read_csv(ruta_direcciones)
df_nodos = pd.read_csv(ruta_nodos)
df_det = pd.read_excel(ruta_detalle)

print(f"Direcciones: {df_dir.shape}")
print(f"Nodos: {df_nodos.shape}")
print(f"Detalle: {df_det.shape}")

# =========================================================
# 2) LIMPIEZA BÁSICA
# =========================================================
# Direcciones
df_dir.columns = [c.strip() for c in df_dir.columns]
df_nodos.columns = [c.strip() for c in df_nodos.columns]
df_det.columns = [c.strip() for c in df_det.columns]

# Convertir fechas relevantes
if "Fecha de Pedido" in df_dir.columns:
    df_dir["Fecha de Pedido"] = convertir_fecha(df_dir["Fecha de Pedido"])

if "Fecha de despacho Solicitada" in df_dir.columns:
    df_dir["Fecha de despacho Solicitada"] = convertir_fecha(df_dir["Fecha de despacho Solicitada"])

# Forzar Número de Orden como string limpio
df_dir["Número de Orden"] = df_dir["Número de Orden"].astype(str).str.strip()
df_det["Número de Orden"] = df_det["Número de Orden"].astype(str).str.strip()

# Coordenadas numéricas
if "lat" in df_dir.columns:
    df_dir["lat"] = pd.to_numeric(df_dir["lat"], errors="coerce")
if "lon" in df_dir.columns:
    df_dir["lon"] = pd.to_numeric(df_dir["lon"], errors="coerce")

if "lat" in df_nodos.columns:
    df_nodos["lat"] = pd.to_numeric(df_nodos["lat"], errors="coerce")
if "lon" in df_nodos.columns:
    df_nodos["lon"] = pd.to_numeric(df_nodos["lon"], errors="coerce")

# =========================================================
# 3) NORMALIZAR DIRECCIONES PARA EL MERGE
# =========================================================
if "direccion_normalizada" not in df_dir.columns:
    raise ValueError("El archivo de direcciones no tiene la columna 'direccion_normalizada'.")

if "direccion_normalizada" not in df_nodos.columns:
    raise ValueError("El archivo de nodos no tiene la columna 'direccion_normalizada'.")

df_dir["direccion_normalizada_key"] = df_dir["direccion_normalizada"].apply(normalizar_texto)
df_nodos["direccion_normalizada_key"] = df_nodos["direccion_normalizada"].apply(normalizar_texto)

# =========================================================
# 4) PREPARAR NODOS
# =========================================================
# Nos quedamos con nodos únicos por key
cols_nodos = [
    "node_id",
    "direccion_original",
    "direccion_normalizada",
    "direccion_normalizada_key",
    "lat",
    "lon",
    "estado",
    "calidad_match",
    "tipo",
    "tiene_coordenadas",
]

cols_nodos_existentes = [c for c in cols_nodos if c in df_nodos.columns]
df_nodos_use = df_nodos[cols_nodos_existentes].copy()

# Si hay duplicados por dirección normalizada, nos quedamos con el primero
df_nodos_use = df_nodos_use.drop_duplicates(subset=["direccion_normalizada_key"], keep="first")

# =========================================================
# 5) MERGE DIRECCIONES + NODOS
# =========================================================
print("Uniendo direcciones con nodos...")

df_base = df_dir.merge(
    df_nodos_use,
    on="direccion_normalizada_key",
    how="left",
    suffixes=("", "_nodo")
)

# Si la dirección geocodificada no tenía lat/lon útiles, podrías usar la del nodo como respaldo
if "lat_nodo" in df_base.columns:
    df_base["lat_final"] = df_base["lat"].fillna(df_base["lat_nodo"])
else:
    df_base["lat_final"] = df_base["lat"]

if "lon_nodo" in df_base.columns:
    df_base["lon_final"] = df_base["lon"].fillna(df_base["lon_nodo"])
else:
    df_base["lon_final"] = df_base["lon"]

# =========================================================
# 6) AGREGAR DETALLE POR ORDEN
# =========================================================
print("Agregando volumen y peso por orden...")

# Revisar columnas requeridas
requeridas_detalle = ["Número de Orden", "Volumen_total_m3", "Peso_total_kg"]
faltantes = [c for c in requeridas_detalle if c not in df_det.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en detalle: {faltantes}")

df_det["Volumen_total_m3"] = pd.to_numeric(df_det["Volumen_total_m3"], errors="coerce").fillna(0)
df_det["Peso_total_kg"] = pd.to_numeric(df_det["Peso_total_kg"], errors="coerce").fillna(0)

# Agregamos por orden
df_det_agg = (
    df_det.groupby("Número de Orden", as_index=False)
    .agg(
        volumen_total_pedido_m3=("Volumen_total_m3", "sum"),
        peso_total_pedido_kg=("Peso_total_kg", "sum"),
        n_lineas_detalle=("SKU", "count")
    )
)

# Merge con base
df_maestra = df_base.merge(
    df_det_agg,
    on="Número de Orden",
    how="left"
)

# Rellenar faltantes numéricos
df_maestra["volumen_total_pedido_m3"] = df_maestra["volumen_total_pedido_m3"].fillna(0)
df_maestra["peso_total_pedido_kg"] = df_maestra["peso_total_pedido_kg"].fillna(0)
df_maestra["n_lineas_detalle"] = df_maestra["n_lineas_detalle"].fillna(0)

# =========================================================
# 7) AÑADIR INFO DEL DEPÓSITO
# =========================================================
depositos = df_nodos[df_nodos["tipo"].astype(str).str.lower() == "deposito"].copy()

if len(depositos) == 0:
    raise ValueError("No se encontró depósito en nodos_master.csv")

deposito = depositos.iloc[0]

df_maestra["node_id_deposito"] = deposito["node_id"]
df_maestra["lat_deposito"] = deposito["lat"]
df_maestra["lon_deposito"] = deposito["lon"]
df_maestra["direccion_deposito"] = deposito["direccion_original"]

# =========================================================
# 8) FILTROS MÍNIMOS RECOMENDADOS
# =========================================================
# Solo pedidos con coordenadas finales válidas
df_maestra["tiene_coords_finales"] = df_maestra["lat_final"].notna() & df_maestra["lon_final"].notna()

# Solo pedidos con node_id encontrado
df_maestra["tiene_node_id"] = df_maestra["node_id"].notna()

# Solo pedidos con detalle
df_maestra["tiene_detalle"] = (
    (df_maestra["volumen_total_pedido_m3"] > 0) |
    (df_maestra["peso_total_pedido_kg"] > 0)
)

# =========================================================
# 9) SELECCIÓN DE COLUMNAS FINALES
# =========================================================
columnas_finales = [
    "Número de Orden",
    "RUT",
    "Nombre cliente",
    "Dirección cliente",
    "Fecha de Pedido",
    "Fecha de despacho Solicitada",
    "Monto Pedido",
    "direccion_normalizada",
    "formatted",
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_pedido_m3",
    "peso_total_pedido_kg",
    "n_lineas_detalle",
    "estado",
    "calidad_match",
    "tipo",
    "tiene_coords_finales",
    "tiene_node_id",
    "tiene_detalle",
    "node_id_deposito",
    "lat_deposito",
    "lon_deposito",
    "direccion_deposito",
]

columnas_finales_existentes = [c for c in columnas_finales if c in df_maestra.columns]
df_maestra_final = df_maestra[columnas_finales_existentes].copy()

# =========================================================
# 10) DIAGNÓSTICOS
# =========================================================
pedidos_sin_node_id = df_maestra_final[df_maestra_final["tiene_node_id"] == False].copy()
pedidos_sin_detalle = df_maestra_final[df_maestra_final["tiene_detalle"] == False].copy()
pedidos_sin_coords = df_maestra_final[df_maestra_final["tiene_coords_finales"] == False].copy()

resumen = pd.DataFrame({
    "indicador": [
        "total_registros_base",
        "pedidos_con_node_id",
        "pedidos_sin_node_id",
        "pedidos_con_detalle",
        "pedidos_sin_detalle",
        "pedidos_con_coords_finales",
        "pedidos_sin_coords_finales",
        "node_id_unicos",
        "fechas_despacho_unicas",
    ],
    "valor": [
        len(df_maestra_final),
        int(df_maestra_final["tiene_node_id"].sum()),
        int((~df_maestra_final["tiene_node_id"]).sum()),
        int(df_maestra_final["tiene_detalle"].sum()),
        int((~df_maestra_final["tiene_detalle"]).sum()),
        int(df_maestra_final["tiene_coords_finales"].sum()),
        int((~df_maestra_final["tiene_coords_finales"]).sum()),
        df_maestra_final["node_id"].nunique(dropna=True),
        df_maestra_final["Fecha de despacho Solicitada"].nunique(dropna=True)
        if "Fecha de despacho Solicitada" in df_maestra_final.columns else np.nan,
    ]
})

# =========================================================
# 11) GUARDAR RESULTADOS
# =========================================================
print("Guardando resultados...")

df_maestra_final.to_csv(ruta_salida_maestra, index=False, encoding="utf-8-sig")
pedidos_sin_node_id.to_csv(ruta_salida_sin_nodo, index=False, encoding="utf-8-sig")
pedidos_sin_detalle.to_csv(ruta_salida_sin_detalle, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_salida_resumen, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="resumen", index=False)
    df_maestra_final.head(5000).to_excel(writer, sheet_name="muestra_maestra", index=False)
    pedidos_sin_node_id.head(5000).to_excel(writer, sheet_name="sin_node_id", index=False)
    pedidos_sin_detalle.head(5000).to_excel(writer, sheet_name="sin_detalle", index=False)
    pedidos_sin_coords.head(5000).to_excel(writer, sheet_name="sin_coords", index=False)

print("Listo.")
print(f"Base maestra: {ruta_salida_maestra}")
print(f"Pedidos sin node_id: {ruta_salida_sin_nodo}")
print(f"Pedidos sin detalle: {ruta_salida_sin_detalle}")
print(f"Resumen Excel: {ruta_salida_resumen}")
print("\nResumen rápido:")
print(resumen)

Leyendo archivos...
Direcciones: (4180, 18)
Nodos: (2256, 9)
Detalle: (53318, 11)
Uniendo direcciones con nodos...
Agregando volumen y peso por orden...
Guardando resultados...
Listo.
Base maestra: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_maestro_clusterizacion.csv
Pedidos sin node_id: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_sin_node_id.csv
Pedidos sin detalle: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_sin_detalle.csv
Resumen Excel: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\resumen_construccion_maestra.xlsx

Resumen rápido:
                    indicador  valor
0        total_registros_base   4180
1         pedidos_con_node_id   4180
2         pedidos_sin_node_id      0
3         pedidos_con_detalle   4180
4         pedidos_sin_detalle      0
5  pedidos_con_coords_finales   4180
6  pedidos_sin_coords_finales      0
7              node_id_unicos   2255
8      fechas_despacho_unicas     35


# ad del anterior

In [4]:
import pandas as pd
import os

ruta_entrada = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_maestro_clusterizacion.csv"
ruta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_maestro_clusterizacion_reducido.csv"

df = pd.read_csv(ruta_entrada)

columnas_utiles = [
    "Número de Orden",
    "Fecha de despacho Solicitada",
    "Monto Pedido",
    "direccion_normalizada",
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_pedido_m3",
    "peso_total_pedido_kg",
    "node_id_deposito",
    "lat_deposito",
    "lon_deposito",
]

# Nos quedamos solo con las que existan realmente
columnas_utiles = [c for c in columnas_utiles if c in df.columns]

df_reducido = df[columnas_utiles].copy()

df_reducido.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print("Archivo reducido guardado en:")
print(ruta_salida)
print("\nColumnas finales:")
print(df_reducido.columns.tolist())
print("\nPrimeras filas:")
print(df_reducido.head())

Archivo reducido guardado en:
C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_maestro_clusterizacion_reducido.csv

Columnas finales:
['Número de Orden', 'Fecha de despacho Solicitada', 'Monto Pedido', 'direccion_normalizada', 'node_id', 'lat_final', 'lon_final', 'volumen_total_pedido_m3', 'peso_total_pedido_kg', 'node_id_deposito', 'lat_deposito', 'lon_deposito']

Primeras filas:
        Número de Orden Fecha de despacho Solicitada  Monto Pedido  \
0  ORD-CL-202612-000001                   2026-12-02        123929   
1  ORD-CL-202612-000002                   2026-12-03        118942   
2  ORD-CL-202612-000003                   2026-12-04        115798   
3  ORD-CL-202612-000004                   2026-12-03        240273   
4  ORD-CL-202612-000005                   2026-12-04        230875   

    direccion_normalizada  node_id  lat_final  lon_final  \
0          Av. Matta 5370      717 -33.457991 -70.642117   
1     Av. La Florida 7678      418 -33.522376 -70.578885 

# Base agregada

In [5]:
import pandas as pd
import numpy as np
import os

# =========================================================
# RUTAS
# =========================================================
ruta_entrada = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_maestro_clusterizacion_reducido.csv"
carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed"

ruta_salida = os.path.join(carpeta_salida, "pedidos_nodo_dia.csv")
ruta_resumen = os.path.join(carpeta_salida, "resumen_pedidos_nodo_dia.xlsx")

# =========================================================
# 1) LEER ARCHIVO
# =========================================================
df = pd.read_csv(ruta_entrada)

# =========================================================
# 2) LIMPIEZA BÁSICA
# =========================================================
df["Fecha de despacho Solicitada"] = pd.to_datetime(
    df["Fecha de despacho Solicitada"], errors="coerce"
)

cols_num = [
    "Monto Pedido",
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_pedido_m3",
    "peso_total_pedido_kg",
    "node_id_deposito",
    "lat_deposito",
    "lon_deposito"
]

for col in cols_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Filtrar filas válidas
df = df[
    df["Fecha de despacho Solicitada"].notna() &
    df["node_id"].notna() &
    df["lat_final"].notna() &
    df["lon_final"].notna()
].copy()

# =========================================================
# 3) AGREGAR POR DIA + NODE_ID
# =========================================================
df_nodo_dia = (
    df.groupby(["Fecha de despacho Solicitada", "node_id"], as_index=False)
      .agg(
          n_pedidos=("Número de Orden", "count"),
          monto_total_nodo_dia=("Monto Pedido", "sum"),
          volumen_total_nodo_dia_m3=("volumen_total_pedido_m3", "sum"),
          peso_total_nodo_dia_kg=("peso_total_pedido_kg", "sum"),
          lat_final=("lat_final", "first"),
          lon_final=("lon_final", "first"),
          direccion_normalizada=("direccion_normalizada", "first"),
          node_id_deposito=("node_id_deposito", "first"),
          lat_deposito=("lat_deposito", "first"),
          lon_deposito=("lon_deposito", "first")
      )
)

# =========================================================
# 4) COLUMNAS ÚTILES
# =========================================================
df_nodo_dia["fecha"] = df_nodo_dia["Fecha de despacho Solicitada"].dt.date

df_nodo_dia["uso_capacidad_vol_pct"] = df_nodo_dia["volumen_total_nodo_dia_m3"] / 45.0
df_nodo_dia["uso_capacidad_peso_pct"] = df_nodo_dia["peso_total_nodo_dia_kg"] / 7500.0

df_nodo_dia["excede_volumen_camion"] = df_nodo_dia["volumen_total_nodo_dia_m3"] > 45
df_nodo_dia["excede_peso_camion"] = df_nodo_dia["peso_total_nodo_dia_kg"] > 7500

# =========================================================
# 5) RESÚMENES
# =========================================================
resumen_general = pd.DataFrame({
    "indicador": [
        "filas_nodo_dia",
        "dias_unicos",
        "node_id_unicos_en_operacion",
        "max_nodos_en_un_dia",
        "promedio_nodos_por_dia",
        "max_pedidos_mismo_nodo_dia",
        "n_filas_excede_volumen_camion",
        "n_filas_excede_peso_camion",
    ],
    "valor": [
        len(df_nodo_dia),
        df_nodo_dia["Fecha de despacho Solicitada"].nunique(),
        df_nodo_dia["node_id"].nunique(),
        df_nodo_dia.groupby("Fecha de despacho Solicitada")["node_id"].nunique().max(),
        df_nodo_dia.groupby("Fecha de despacho Solicitada")["node_id"].nunique().mean(),
        df_nodo_dia["n_pedidos"].max(),
        int(df_nodo_dia["excede_volumen_camion"].sum()),
        int(df_nodo_dia["excede_peso_camion"].sum()),
    ]
})

resumen_por_dia = (
    df_nodo_dia.groupby("Fecha de despacho Solicitada", as_index=False)
    .agg(
        nodos_dia=("node_id", "nunique"),
        pedidos_dia=("n_pedidos", "sum"),
        volumen_total_dia_m3=("volumen_total_nodo_dia_m3", "sum"),
        peso_total_dia_kg=("peso_total_nodo_dia_kg", "sum"),
        monto_total_dia=("monto_total_nodo_dia", "sum")
    )
)

resumen_por_dia["lb_rutas_por_volumen"] = np.ceil(resumen_por_dia["volumen_total_dia_m3"] / 45.0)
resumen_por_dia["lb_rutas_por_peso"] = np.ceil(resumen_por_dia["peso_total_dia_kg"] / 7500.0)
resumen_por_dia["lb_rutas_teorica"] = resumen_por_dia[
    ["lb_rutas_por_volumen", "lb_rutas_por_peso"]
].max(axis=1)

# =========================================================
# 6) GUARDAR
# =========================================================
df_nodo_dia.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_resumen, engine="openpyxl") as writer:
    resumen_general.to_excel(writer, sheet_name="resumen_general", index=False)
    resumen_por_dia.to_excel(writer, sheet_name="resumen_por_dia", index=False)
    df_nodo_dia.head(5000).to_excel(writer, sheet_name="muestra_nodo_dia", index=False)

print("Listo.")
print("Archivo generado:", ruta_salida)
print("Resumen generado:", ruta_resumen)
print("\nResumen general:")
print(resumen_general)

Listo.
Archivo generado: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_nodo_dia.csv
Resumen generado: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\resumen_pedidos_nodo_dia.xlsx

Resumen general:
                       indicador        valor
0                 filas_nodo_dia  4087.000000
1                    dias_unicos    35.000000
2    node_id_unicos_en_operacion  2255.000000
3            max_nodos_en_un_dia   177.000000
4         promedio_nodos_por_dia   116.771429
5     max_pedidos_mismo_nodo_dia     3.000000
6  n_filas_excede_volumen_camion     0.000000
7     n_filas_excede_peso_camion     0.000000


# Código para preparar base de clusterización

In [6]:
import pandas as pd
import numpy as np
import os

# =========================================================
# RUTAS
# =========================================================
ruta_entrada = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_nodo_dia.csv"
carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed"

ruta_salida = os.path.join(carpeta_salida, "pedidos_nodo_dia_cluster_base.csv")
ruta_resumen = os.path.join(carpeta_salida, "resumen_cluster_base.xlsx")

# =========================================================
# FUNCION HAVERSINE
# =========================================================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# =========================================================
# 1) LEER ARCHIVO
# =========================================================
df = pd.read_csv(ruta_entrada)

# Convertir numéricos
cols_num = [
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_nodo_dia_m3",
    "peso_total_nodo_dia_kg",
    "n_pedidos",
    "node_id_deposito",
    "lat_deposito",
    "lon_deposito",
]
for col in cols_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["Fecha de despacho Solicitada"] = pd.to_datetime(
    df["Fecha de despacho Solicitada"], errors="coerce"
)

# =========================================================
# 2) DISTANCIA AL DEPOSITO
# =========================================================
df["dist_deposito_km"] = haversine_km(
    df["lat_deposito"],
    df["lon_deposito"],
    df["lat_final"],
    df["lon_final"]
)

# =========================================================
# 3) ANGULO RESPECTO AL DEPOSITO
# =========================================================
dy = df["lat_final"] - df["lat_deposito"]
dx = df["lon_final"] - df["lon_deposito"]

df["angulo_rad"] = np.arctan2(dy, dx)
df["angulo_grados"] = np.degrees(df["angulo_rad"])
df["angulo_grados"] = (df["angulo_grados"] + 360) % 360

# =========================================================
# 4) COLUMNAS UTILES EXTRA
# =========================================================
df["fecha"] = pd.to_datetime(df["Fecha de despacho Solicitada"]).dt.date

df["carga_relativa_max"] = df[
    ["uso_capacidad_vol_pct", "uso_capacidad_peso_pct"]
].max(axis=1)

# =========================================================
# 5) ORDEN SUGERIDO PARA SWEEP
# =========================================================
df = df.sort_values(
    by=["Fecha de despacho Solicitada", "angulo_grados", "dist_deposito_km"],
    ascending=[True, True, True]
).reset_index(drop=True)

# =========================================================
# 6) RESUMENES
# =========================================================
resumen_general = pd.DataFrame({
    "indicador": [
        "filas_cluster_base",
        "dist_min_km",
        "dist_prom_km",
        "dist_max_km",
        "angulo_min",
        "angulo_max"
    ],
    "valor": [
        len(df),
        df["dist_deposito_km"].min(),
        df["dist_deposito_km"].mean(),
        df["dist_deposito_km"].max(),
        df["angulo_grados"].min(),
        df["angulo_grados"].max(),
    ]
})

resumen_por_dia = (
    df.groupby("Fecha de despacho Solicitada", as_index=False)
    .agg(
        nodos_dia=("node_id", "count"),
        dist_prom_km=("dist_deposito_km", "mean"),
        dist_max_km=("dist_deposito_km", "max"),
        carga_max_prom=("carga_relativa_max", "mean"),
        volumen_total_dia_m3=("volumen_total_nodo_dia_m3", "sum"),
        peso_total_dia_kg=("peso_total_nodo_dia_kg", "sum")
    )
)

# =========================================================
# 7) GUARDAR
# =========================================================
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_resumen, engine="openpyxl") as writer:
    resumen_general.to_excel(writer, sheet_name="resumen_general", index=False)
    resumen_por_dia.to_excel(writer, sheet_name="resumen_por_dia", index=False)
    df.head(5000).to_excel(writer, sheet_name="muestra_cluster_base", index=False)

print("Listo.")
print("Archivo generado:", ruta_salida)
print("Resumen generado:", ruta_resumen)
print("\nResumen general:")
print(resumen_general)

Listo.
Archivo generado: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\pedidos_nodo_dia_cluster_base.csv
Resumen generado: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\processed\resumen_cluster_base.xlsx

Resumen general:
            indicador        valor
0  filas_cluster_base  4087.000000
1         dist_min_km     0.591290
2        dist_prom_km     6.884286
3         dist_max_km    18.906424
4          angulo_min     0.127339
5          angulo_max   359.448189


# Código de clusterización Sweep

In [35]:
import pandas as pd
import numpy as np
import os

# =========================================================
# RUTAS
# =========================================================
ruta_entrada = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_cluster_base.csv"
carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters"

ruta_salida = os.path.join(carpeta_salida, "pedidos_nodo_dia_clusterizados.csv")
ruta_resumen = os.path.join(carpeta_salida, "resumen_clusterizacion_sweep.xlsx")

# =========================================================
# PARÁMETROS DE CLUSTERIZACIÓN
# =========================================================
CAP_VOL_CLUSTER = 40.0      # m3
CAP_PESO_CLUSTER = 6500.0   # kg
MAX_NODOS_CLUSTER = 10000000      # máximo de nodos por cluster

# =========================================================
# 1) LEER BASE
# =========================================================
df = pd.read_csv(ruta_entrada)

df["Fecha de despacho Solicitada"] = pd.to_datetime(
    df["Fecha de despacho Solicitada"], errors="coerce"
)

# Asegurar tipos
cols_num = [
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_nodo_dia_m3",
    "peso_total_nodo_dia_kg",
    "dist_deposito_km",
    "angulo_grados",
    "n_pedidos",
]
for col in cols_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Filtrar filas válidas
df = df[
    df["Fecha de despacho Solicitada"].notna() &
    df["node_id"].notna() &
    df["volumen_total_nodo_dia_m3"].notna() &
    df["peso_total_nodo_dia_kg"].notna() &
    df["angulo_grados"].notna() &
    df["dist_deposito_km"].notna()
].copy()

# Orden para sweep
df = df.sort_values(
    by=["Fecha de despacho Solicitada", "angulo_grados", "dist_deposito_km"],
    ascending=[True, True, True]
).reset_index(drop=True)

# =========================================================
# 2) CLUSTERIZACIÓN SWEEP
# =========================================================
registros = []
cluster_global = 0

for fecha, grupo in df.groupby("Fecha de despacho Solicitada", sort=True):
    grupo = grupo.sort_values(["angulo_grados", "dist_deposito_km"]).copy()

    cluster_dia = 1
    cluster_global += 1

    vol_acum = 0.0
    peso_acum = 0.0
    nodos_acum = 0

    for _, row in grupo.iterrows():
        vol = float(row["volumen_total_nodo_dia_m3"])
        peso = float(row["peso_total_nodo_dia_kg"])

        rompe_cluster = (
            (vol_acum + vol > CAP_VOL_CLUSTER) or
            (peso_acum + peso > CAP_PESO_CLUSTER) or
            (nodos_acum + 1 > MAX_NODOS_CLUSTER)
        )

        if rompe_cluster:
            cluster_dia += 1
            cluster_global += 1
            vol_acum = 0.0
            peso_acum = 0.0
            nodos_acum = 0

        vol_acum += vol
        peso_acum += peso
        nodos_acum += 1

        row_dict = row.to_dict()
        row_dict["cluster_id_global"] = int(cluster_global)
        row_dict["cluster_id_dia"] = int(cluster_dia)
        row_dict["vol_acum_cluster"] = vol_acum
        row_dict["peso_acum_cluster"] = peso_acum
        row_dict["nodos_acum_cluster"] = nodos_acum

        registros.append(row_dict)

df_clusters = pd.DataFrame(registros)

# =========================================================
# 3) RESUMEN POR CLUSTER
# =========================================================
resumen_clusters = (
    df_clusters.groupby(["Fecha de despacho Solicitada", "cluster_id_dia"], as_index=False)
    .agg(
        n_nodos=("node_id", "count"),
        n_pedidos=("n_pedidos", "sum"),
        volumen_cluster_m3=("volumen_total_nodo_dia_m3", "sum"),
        peso_cluster_kg=("peso_total_nodo_dia_kg", "sum"),
        dist_prom_deposito_km=("dist_deposito_km", "mean"),
        dist_max_deposito_km=("dist_deposito_km", "max"),
        angulo_min=("angulo_grados", "min"),
        angulo_max=("angulo_grados", "max")
    )
)

resumen_clusters["uso_vol_pct"] = resumen_clusters["volumen_cluster_m3"] / 45.0
resumen_clusters["uso_peso_pct"] = resumen_clusters["peso_cluster_kg"] / 7500.0

# =========================================================
# 4) RESUMEN GENERAL
# =========================================================
resumen_general = pd.DataFrame({
    "indicador": [
        "filas_clusterizadas",
        "dias_unicos",
        "clusters_totales",
        "max_clusters_en_un_dia",
        "prom_clusters_por_dia",
        "prom_nodos_por_cluster",
        "max_nodos_por_cluster",
        "prom_uso_vol_pct",
        "prom_uso_peso_pct"
    ],
    "valor": [
        len(df_clusters),
        df_clusters["Fecha de despacho Solicitada"].nunique(),
        resumen_clusters.shape[0],
        resumen_clusters.groupby("Fecha de despacho Solicitada")["cluster_id_dia"].nunique().max(),
        resumen_clusters.groupby("Fecha de despacho Solicitada")["cluster_id_dia"].nunique().mean(),
        resumen_clusters["n_nodos"].mean(),
        resumen_clusters["n_nodos"].max(),
        resumen_clusters["uso_vol_pct"].mean(),
        resumen_clusters["uso_peso_pct"].mean()
    ]
})

resumen_por_dia = (
    resumen_clusters.groupby("Fecha de despacho Solicitada", as_index=False)
    .agg(
        clusters_dia=("cluster_id_dia", "nunique"),
        nodos_dia=("n_nodos", "sum"),
        pedidos_dia=("n_pedidos", "sum"),
        volumen_total_dia_m3=("volumen_cluster_m3", "sum"),
        peso_total_dia_kg=("peso_cluster_kg", "sum")
    )
)

# =========================================================
# 5) GUARDAR
# =========================================================
df_clusters.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_resumen, engine="openpyxl") as writer:
    resumen_general.to_excel(writer, sheet_name="resumen_general", index=False)
    resumen_por_dia.to_excel(writer, sheet_name="resumen_por_dia", index=False)
    resumen_clusters.to_excel(writer, sheet_name="resumen_clusters", index=False)
    df_clusters.head(5000).to_excel(writer, sheet_name="muestra_clusterizada", index=False)

print("Listo.")
print("Archivo clusterizado:", ruta_salida)
print("Resumen:", ruta_resumen)
print("\nResumen general:")
print(resumen_general)

Listo.
Archivo clusterizado: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_clusterizados.csv
Resumen: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\resumen_clusterizacion_sweep.xlsx

Resumen general:
                indicador        valor
0     filas_clusterizadas  4087.000000
1             dias_unicos    35.000000
2        clusters_totales   135.000000
3  max_clusters_en_un_dia     6.000000
4   prom_clusters_por_dia     3.857143
5  prom_nodos_por_cluster    30.274074
6   max_nodos_por_cluster    47.000000
7        prom_uso_vol_pct     0.743490
8       prom_uso_peso_pct     0.652203


# Asignar ventana AM/PM

In [36]:
import pandas as pd
import numpy as np
import os

# =========================================================
# RUTAS
# =========================================================
ruta_clusters = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_clusterizados.csv"
carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters"

ruta_salida = os.path.join(carpeta_salida, "pedidos_nodo_dia_clusterizados_am_pm.csv")
ruta_resumen = os.path.join(carpeta_salida, "resumen_am_pm.xlsx")

# =========================================================
# 1) LEER
# =========================================================
df = pd.read_csv(ruta_clusters)
df["Fecha de despacho Solicitada"] = pd.to_datetime(df["Fecha de despacho Solicitada"], errors="coerce")

# =========================================================
# 2) RESUMEN POR CLUSTER
# =========================================================
resumen_clusters = (
    df.groupby(["Fecha de despacho Solicitada", "cluster_id_dia"], as_index=False)
    .agg(
        n_nodos=("node_id", "count"),
        n_pedidos=("n_pedidos", "sum"),
        volumen_cluster_m3=("volumen_total_nodo_dia_m3", "sum"),
        peso_cluster_kg=("peso_total_nodo_dia_kg", "sum"),
        dist_prom_deposito_km=("dist_deposito_km", "mean"),
        dist_max_deposito_km=("dist_deposito_km", "max")
    )
)

resumen_clusters["uso_vol_pct"] = resumen_clusters["volumen_cluster_m3"] / 45.0
resumen_clusters["uso_peso_pct"] = resumen_clusters["peso_cluster_kg"] / 7500.0
resumen_clusters["carga_relativa"] = resumen_clusters[["uso_vol_pct", "uso_peso_pct"]].max(axis=1)

# =========================================================
# 3) SCORE DE EXIGENCIA
# =========================================================
resumen_clusters["score_exigencia"] = (
    0.5 * resumen_clusters["dist_prom_deposito_km"] +
    0.3 * resumen_clusters["n_nodos"] +
    0.2 * (100 * resumen_clusters["carga_relativa"])
)

# =========================================================
# 4) ASIGNAR AM / PM POR DIA
# =========================================================
ventanas = []

for fecha, grupo in resumen_clusters.groupby("Fecha de despacho Solicitada", sort=True):
    grupo = grupo.sort_values(
        by=["score_exigencia", "dist_prom_deposito_km", "n_nodos"],
        ascending=[False, False, False]
    ).copy()

    n = len(grupo)
    n_am = int(np.ceil(n / 2))

    grupo["ventana"] = "PM"
    grupo.iloc[:n_am, grupo.columns.get_loc("ventana")] = "AM"

    ventanas.append(grupo)

resumen_clusters_am_pm = pd.concat(ventanas, ignore_index=True)

# =========================================================
# 5) PEGAR VENTANA A NIVEL NODO
# =========================================================
df_final = df.merge(
    resumen_clusters_am_pm[
        ["Fecha de despacho Solicitada", "cluster_id_dia", "ventana", "score_exigencia"]
    ],
    on=["Fecha de despacho Solicitada", "cluster_id_dia"],
    how="left"
)

# =========================================================
# 6) RESUMEN
# =========================================================
resumen_ventanas = (
    resumen_clusters_am_pm.groupby(["Fecha de despacho Solicitada", "ventana"], as_index=False)
    .agg(
        clusters=("cluster_id_dia", "nunique"),
        nodos=("n_nodos", "sum"),
        pedidos=("n_pedidos", "sum"),
        volumen_m3=("volumen_cluster_m3", "sum"),
        peso_kg=("peso_cluster_kg", "sum")
    )
)

# =========================================================
# 7) GUARDAR
# =========================================================
df_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_resumen, engine="openpyxl") as writer:
    resumen_clusters_am_pm.to_excel(writer, sheet_name="clusters_am_pm", index=False)
    resumen_ventanas.to_excel(writer, sheet_name="resumen_ventanas", index=False)
    df_final.head(5000).to_excel(writer, sheet_name="muestra_final", index=False)

print("Listo.")
print("Archivo final:", ruta_salida)
print("Resumen:", ruta_resumen)
print("\nClusters por ventana:")
print(resumen_ventanas.head(20))

Listo.
Archivo final: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_clusterizados_am_pm.csv
Resumen: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\resumen_am_pm.xlsx

Clusters por ventana:
   Fecha de despacho Solicitada ventana  clusters  nodos  pedidos  volumen_m3  \
0                    2026-12-02      AM         1     14       14   13.228068   
1                    2026-12-03      AM         2     64       65   79.121398   
2                    2026-12-03      PM         1     20       20   15.208348   
3                    2026-12-04      AM         2     69       72   77.897473   
4                    2026-12-04      PM         2     33       35   40.739020   
5                    2026-12-05      AM         2     68       69   74.222534   
6                    2026-12-05      PM         2     47       49   63.278241   
7                    2026-12-06      AM         2     79       80   77.982932   
8                    2026-12-06      PM  

# Código para estimar tiempo por cluster

In [37]:
import pandas as pd
import numpy as np
import os

# =========================================================
# RUTAS
# =========================================================
ruta_entrada = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_clusterizados_am_pm.csv"
carpeta_salida = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters"

ruta_salida_clusters = os.path.join(carpeta_salida, "clusters_con_tiempo_estimado.csv")
ruta_resumen = os.path.join(carpeta_salida, "resumen_clusters_tiempo_estimado.xlsx")

# =========================================================
# PARAMETROS
# =========================================================
ALPHA_KM_POR_NODO = 0.8        # km internos aproximados por nodo
VELOCIDAD_KMH = 25.0           # velocidad promedio urbana
TIEMPO_SERVICIO_MIN = 10.0     # minutos por nodo

# umbrales para clasificar factibilidad media jornada
UMBRAL_OK_H = 4.5
UMBRAL_TENSO_H = 5.5

# =========================================================
# 1) LEER
# =========================================================
df = pd.read_csv(ruta_entrada)
df["Fecha de despacho Solicitada"] = pd.to_datetime(df["Fecha de despacho Solicitada"], errors="coerce")

# =========================================================
# 2) RESUMEN POR CLUSTER
# =========================================================
clusters = (
    df.groupby(["Fecha de despacho Solicitada", "cluster_id_dia", "ventana"], as_index=False)
    .agg(
        n_nodos=("node_id", "count"),
        n_pedidos=("n_pedidos", "sum"),
        volumen_cluster_m3=("volumen_total_nodo_dia_m3", "sum"),
        peso_cluster_kg=("peso_total_nodo_dia_kg", "sum"),
        dist_prom_deposito_km=("dist_deposito_km", "mean"),
        dist_max_deposito_km=("dist_deposito_km", "max"),
    )
)

# =========================================================
# 3) ESTIMACION DE DISTANCIA Y TIEMPO
# =========================================================
clusters["distancia_estimada_ruta_km"] = (
    2 * clusters["dist_prom_deposito_km"] +
    ALPHA_KM_POR_NODO * clusters["n_nodos"]
)

clusters["tiempo_viaje_estimado_h"] = (
    clusters["distancia_estimada_ruta_km"] / VELOCIDAD_KMH
)

clusters["tiempo_servicio_estimado_h"] = (
    clusters["n_nodos"] * TIEMPO_SERVICIO_MIN / 60.0
)

clusters["tiempo_total_estimado_h"] = (
    clusters["tiempo_viaje_estimado_h"] +
    clusters["tiempo_servicio_estimado_h"]
)

# =========================================================
# 4) CLASIFICACION
# =========================================================
def clasificar_cluster(t):
    if t <= UMBRAL_OK_H:
        return "OK"
    elif t <= UMBRAL_TENSO_H:
        return "TENSO"
    return "EXCESIVO"

clusters["estado_tiempo_cluster"] = clusters["tiempo_total_estimado_h"].apply(clasificar_cluster)

# también útil: uso de capacidad
clusters["uso_vol_pct"] = clusters["volumen_cluster_m3"] / 45.0
clusters["uso_peso_pct"] = clusters["peso_cluster_kg"] / 7500.0

# =========================================================
# 5) RESUMEN POR VENTANA
# =========================================================
resumen_ventana = (
    clusters.groupby(["Fecha de despacho Solicitada", "ventana"], as_index=False)
    .agg(
        clusters_ventana=("cluster_id_dia", "nunique"),
        nodos_ventana=("n_nodos", "sum"),
        pedidos_ventana=("n_pedidos", "sum"),
        tiempo_total_ventana_h=("tiempo_total_estimado_h", "sum"),
        tiempo_prom_cluster_h=("tiempo_total_estimado_h", "mean"),
        clusters_ok=("estado_tiempo_cluster", lambda s: (s == "OK").sum()),
        clusters_tensos=("estado_tiempo_cluster", lambda s: (s == "TENSO").sum()),
        clusters_excesivos=("estado_tiempo_cluster", lambda s: (s == "EXCESIVO").sum()),
    )
)

# =========================================================
# 6) GUARDAR
# =========================================================
clusters.to_csv(ruta_salida_clusters, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ruta_resumen, engine="openpyxl") as writer:
    clusters.to_excel(writer, sheet_name="clusters_tiempo", index=False)
    resumen_ventana.to_excel(writer, sheet_name="resumen_ventana", index=False)

print("Listo.")
print("Archivo:", ruta_salida_clusters)
print("Resumen:", ruta_resumen)
print("\nResumen por ventana:")
print(resumen_ventana.head(20))

Listo.
Archivo: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\clusters_con_tiempo_estimado.csv
Resumen: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\resumen_clusters_tiempo_estimado.xlsx

Resumen por ventana:
   Fecha de despacho Solicitada ventana  clusters_ventana  nodos_ventana  \
0                    2026-12-02      AM                 1             14   
1                    2026-12-03      AM                 2             64   
2                    2026-12-03      PM                 1             20   
3                    2026-12-04      AM                 2             69   
4                    2026-12-04      PM                 2             33   
5                    2026-12-05      AM                 2             68   
6                    2026-12-05      PM                 2             47   
7                    2026-12-06      AM                 2             79   
8                    2026-12-06      PM                 2             34   
9     

# diagnostico

In [38]:
import pandas as pd

ruta = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\clusters_con_tiempo_estimado.csv"
df = pd.read_csv(ruta)

print(df["estado_tiempo_cluster"].value_counts())

print("\nClusters más largos:")
print(
    df.sort_values("tiempo_total_estimado_h", ascending=False)[
        ["Fecha de despacho Solicitada", "cluster_id_dia", "ventana", "n_nodos", "tiempo_total_estimado_h", "estado_tiempo_cluster"]
    ].head(15)
)

estado_tiempo_cluster
EXCESIVO    106
OK           22
TENSO         7
Name: count, dtype: int64

Clusters más largos:
    Fecha de despacho Solicitada  cluster_id_dia ventana  n_nodos  \
127                   2027-01-02               2      AM       47   
86                    2026-12-23               1      AM       44   
84                    2026-12-22               3      AM       45   
87                    2026-12-23               2      AM       44   
63                    2026-12-18               1      AM       43   
113                   2026-12-29               2      AM       43   
108                   2026-12-28               1      AM       43   
35                    2026-12-12               1      AM       43   
109                   2026-12-28               2      AM       42   
24                    2026-12-09               1      AM       42   
12                    2026-12-06               1      AM       41   
72                    2026-12-19               4      

# Limpieza

In [39]:
import pandas as pd

ruta_in = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\pedidos_nodo_dia_clusterizados_am_pm.csv"
ruta_out = r"C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\input_ruteo_companero.csv"

df = pd.read_csv(ruta_in)

columnas = [
    "Fecha de despacho Solicitada",
    "cluster_id_dia",
    "ventana",
    "node_id",
    "lat_final",
    "lon_final",
    "volumen_total_nodo_dia_m3",
    "peso_total_nodo_dia_kg",
    "n_pedidos",
    "node_id_deposito",
    "lat_deposito",
    "lon_deposito"
]

columnas = [c for c in columnas if c in df.columns]
df_out = df[columnas].copy()

df_out.to_csv(ruta_out, index=False, encoding="utf-8-sig")

print("Archivo listo para ruteo:", ruta_out)

Archivo listo para ruteo: C:\Users\thoma\Documents\GitHub\Capstone-p1\data\clusters\input_ruteo_companero.csv
